# 📊 A/B Testing Analysis Project


## 1. Setup and Data Loading
- Import libraries
- Load raw data

In [ ]:
# Import necessary libraries
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from statsmodels.stats.proportion import proportions_ztest, proportion_confint
import warnings
warnings.filterwarnings('ignore')

# Set visualization style
sns.set(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# Load the dataset
df = pd.read_csv('../data/ab_test_dataset.csv')

# Quick overview of the data
print("First 5 rows:")
print(df.head())
print("\nDataset Info:")
print(df.info())
print("\nMissing Values:")
print(df.isnull().sum())

## 2. Data Cleaning and Validation
- Handle duplicates
- Handle missing values
- Standardize categorical columns
- Handle outliers
- Convert date columns

In [ ]:
# 1. Check and remove duplicate rows
print(f"Duplicate rows before: {df.duplicated().sum()}")
df = df.drop_duplicates()
print(f"Duplicate rows after: {df.duplicated().sum()}")

# 2. Handle missing values
print("\nMissing values per column:")
print(df.isnull().sum())

# Drop rows with missing values in 'click_time' column
df = df.dropna(subset=['click_time'])

# 3. Standardize categorical columns
# Standardize 'group' column
df['group'] = df['group'].str.strip().str.lower()
df['group'] = df['group'].map({
    'con': 'control',
    'control': 'control',
    'exp': 'treatment',
    'treatment': 'treatment'
})

# Standardize 'device_type' column
df['device_type'] = df['device_type'].str.strip().str.lower()
df['device_type'] = df['device_type'].map({
    'mobile': 'mobile',
    'desktop': 'desktop',
    'desk': 'desktop'
})

# Clean 'referral_source' column (remove extra spaces)
df['referral_source'] = df['referral_source'].str.strip().str.lower()

# 4. Handle outliers in 'session_time' using IQR method
Q1 = df['session_time'].quantile(0.25)
Q3 = df['session_time'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print(f"\nOutliers in session_time before: {len(df[(df['session_time'] < lower_bound) | (df['session_time'] > upper_bound)])}")
df = df[(df['session_time'] >= lower_bound) & (df['session_time'] <= upper_bound)]
print(f"Outliers in session_time after: {len(df[(df['session_time'] < lower_bound) | (df['session_time'] > upper_bound)])}")

# 5. Convert date column
df['click_time'] = pd.to_datetime(df['click_time'])

# Verify cleaned dataset
print("\nCleaned Dataset Info:")
print(df.info())

## 3. Exploratory Data Analysis (EDA)
- Group distribution
- CTR by group
- Device type analysis
- Session time analysis

In [ ]:
os.makedirs('../reports/images/eda', exist_ok=True)

# 1. Distribution of groups
print("Group Distribution:")
print(df['group'].value_counts(normalize=True))

# Visualize group distribution
fig, ax = plt.subplots(figsize=(8, 6))
sns.countplot(x='group', data=df, palette='Set2')
ax.set_title('Distribution of Users by Group', fontsize=14)
ax.set_xlabel('Group')
ax.set_ylabel('Count')
plt.savefig('../reports/images/eda/group_distribution.png', dpi=300, bbox_inches='tight')
plt.close()
print(" Saved: eda/group_distribution.png")

# 2. Click-through rate (CTR) by group
ctr_by_group = df.groupby('group')['click'].mean()
print(f"\nClick-Through Rate by Group:\n{ctr_by_group}")

# Visualize CTR comparison
fig, ax = plt.subplots(figsize=(8, 6))
sns.barplot(x='group', y='click', data=df, ci=95, palette='Set2')
ax.set_title('Click-Through Rate by Group with 95% CI', fontsize=14)
ax.set_ylabel('Click-Through Rate')
ax.set_xlabel('Group')
ax.set_ylim(0, 0.6)
plt.savefig('../reports/images/eda/ctr_by_group.png', dpi=300, bbox_inches='tight')
plt.close()
print(" Saved: eda/ctr_by_group.png")

# 3. Distribution by device type
print("\nDevice Type Distribution:")
print(df['device_type'].value_counts(normalize=True))

# Cross-tabulation: Group vs Device Type
device_crosstab = pd.crosstab(df['group'], df['device_type'], normalize='index')
print("\nDevice Type Distribution by Group:")
print(device_crosstab)

# Visualize device distribution by group
fig, ax = plt.subplots(figsize=(8, 6))
device_crosstab.plot(kind='bar', stacked=True, ax=ax, color=['#FF6B6B', '#4ECDC4'])
ax.set_title('Device Type Distribution by Group', fontsize=14)
ax.set_xlabel('Group')
ax.set_ylabel('Proportion')
ax.legend(title='Device Type')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
plt.savefig('../reports/images/eda/device_distribution.png', dpi=300, bbox_inches='tight')
plt.close()
print(" Saved: eda/device_distribution.png")

# 4. Session time analysis
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Histogram
sns.histplot(df[df['group'] == 'control']['session_time'], label='Control', kde=True, color='blue', ax=ax1)
sns.histplot(df[df['group'] == 'treatment']['session_time'], label='Treatment', kde=True, color='red', ax=ax1)
ax1.set_title('Session Time Distribution by Group', fontsize=14)
ax1.set_xlabel('Session Time (minutes)')
ax1.set_ylabel('Frequency')
ax1.legend()

# Boxplot
sns.boxplot(x='group', y='session_time', data=df, palette='Set2', ax=ax2)
ax2.set_title('Session Time Box Plot by Group', fontsize=14)
ax2.set_xlabel('Group')
ax2.set_ylabel('Session Time (minutes)')

plt.tight_layout()
plt.savefig('../reports/images/eda/session_time_analysis.png', dpi=300, bbox_inches='tight')
plt.close()
print(" Saved: eda/session_time_analysis.png")

print("\n All EDA visualizations saved in '../reports/images/eda/'")

## 4. Hypothesis Testing (Z-Test)
- Define hypotheses
- Perform Z-test
- Calculate Lift

In [ ]:
# Define hypotheses:
# H0: CTR_treatment <= CTR_control (new version has no positive effect)
# H1: CTR_treatment > CTR_control (new version has positive effect)
# Significance level: alpha = 0.05

# Prepare data for Z-test
conversions = df.groupby('group')['click'].sum()
visitors = df.groupby('group')['click'].count()

print(f"Conversions (clicks) - Control: {conversions['control']}, Treatment: {conversions['treatment']}")
print(f"Visitors - Control: {visitors['control']}, Treatment: {visitors['treatment']}")

# Perform two-sample proportion Z-test
counts = [conversions['treatment'], conversions['control']]
nobs = [visitors['treatment'], visitors['control']]
z_stat, p_value = proportions_ztest(counts, nobs, alternative='larger')

print(f"\nZ-statistic: {z_stat:.4f}")
print(f"P-value: {p_value:.4f}")

# Decision
alpha = 0.05
if p_value < alpha:
    print(f"\nResult: Reject null hypothesis (p = {p_value:.4f})")
    print("Conclusion: The new design has a statistically significant positive effect on CTR.")
else:
    print(f"\nResult: Fail to reject null hypothesis (p = {p_value:.4f})")
    print("Conclusion: No statistically significant positive effect detected.")

# Calculate confidence interval for treatment group CTR
ci_lower, ci_upper = proportion_confint(
    conversions['treatment'], 
    visitors['treatment'], 
    alpha=alpha, 
    method='normal'
)
print(f"\n95% Confidence Interval for Treatment CTR: [{ci_lower:.4f}, {ci_upper:.4f}]")

# Calculate Lift (percentage improvement)
control_ctr = conversions['control'] / visitors['control']
treatment_ctr = conversions['treatment'] / visitors['treatment']
lift = (treatment_ctr - control_ctr) / control_ctr * 100
print(f"\nControl CTR: {control_ctr:.4f}")
print(f"Treatment CTR: {treatment_ctr:.4f}")
print(f"Lift: {lift:.2f}%")

## 5. Segmentation Analysis
- Device-based analysis
- Referral source analysis

In [ ]:
# 1. Device-based analysis
print("\n" + "="*50)
print("SEGMENTATION ANALYSIS BY DEVICE TYPE")
print("="*50)

# Mobile users
df_mobile = df[df['device_type'] == 'mobile']
conv_mobile = df_mobile.groupby('group')['click'].sum()
vis_mobile = df_mobile.groupby('group')['click'].count()

z_mobile, p_mobile = proportions_ztest(
    [conv_mobile['treatment'], conv_mobile['control']],
    [vis_mobile['treatment'], vis_mobile['control']],
    alternative='larger'
)

print(f"\nMobile Users:")
print(f"  Z-statistic: {z_mobile:.4f}")
print(f"  P-value: {p_mobile:.4f}")
print(f"  Lift: {((conv_mobile['treatment']/vis_mobile['treatment']) - (conv_mobile['control']/vis_mobile['control'])) / (conv_mobile['control']/vis_mobile['control']) * 100:.2f}%")

# Desktop users
df_desktop = df[df['device_type'] == 'desktop']
conv_desktop = df_desktop.groupby('group')['click'].sum()
vis_desktop = df_desktop.groupby('group')['click'].count()

z_desktop, p_desktop = proportions_ztest(
    [conv_desktop['treatment'], conv_desktop['control']],
    [vis_desktop['treatment'], vis_desktop['control']],
    alternative='larger'
)

print(f"\nDesktop Users:")
print(f"  Z-statistic: {z_desktop:.4f}")
print(f"  P-value: {p_desktop:.4f}")
print(f"  Lift: {((conv_desktop['treatment']/vis_desktop['treatment']) - (conv_desktop['control']/vis_desktop['control'])) / (conv_desktop['control']/vis_desktop['control']) * 100:.2f}%")

# 2. Referral source analysis
print("\n" + "="*50)
print("SEGMENTATION ANALYSIS BY REFERRAL SOURCE")
print("="*50)

referral_sources = df['referral_source'].unique()
for source in referral_sources:
    if pd.isna(source) or source == '':
        continue
    
    df_source = df[df['referral_source'] == source]
    if len(df_source) > 100:  # Ensure enough samples for meaningful test
        conv_source = df_source.groupby('group')['click'].sum()
        vis_source = df_source.groupby('group')['click'].count()
        
        if all(v > 0 for v in vis_source):
            z_source, p_source = proportions_ztest(
                [conv_source['treatment'], conv_source['control']],
                [vis_source['treatment'], vis_source['control']],
                alternative='larger'
            )
            print(f"\n{source.upper()}:")
            print(f"  Z-statistic: {z_source:.4f}")
            print(f"  P-value: {p_source:.4f}")

## 6. Visualization and Reporting
- Key charts
- Export cleaned data

In [ ]:
# ==========================================
# Comprehensive Visualization for Final Report 
# ==========================================

# Create directory for saving images
os.makedirs('../reports/images/final', exist_ok=True)

# PART 1: Combined Figure (4 plots in one image)

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('A/B Test Analysis Results', fontsize=16, y=0.98)

# Plot 1: CTR comparison
ax1 = axes[0, 0]
ctr_data = df.groupby('group')['click'].agg(['mean', 'std']).reset_index()
sns.barplot(x='group', y='mean', data=ctr_data, ax=ax1, palette='Set2')
ax1.set_title('Click-Through Rate by Group', fontsize=14)
ax1.set_ylabel('CTR')
ax1.set_xlabel('Group')
ax1.set_ylim(0, 0.6)
for i, row in ctr_data.iterrows():
    ax1.text(i, row['mean'] + 0.02, f"{row['mean']:.3f}", ha='center')

# Plot 2: Confidence intervals
ax2 = axes[0, 1]
groups = ['Control', 'Treatment']
ctr_values = [control_ctr, treatment_ctr]
ci_lower_values = [
    proportion_confint(conversions['control'], visitors['control'], alpha=0.05, method='normal')[0],
    proportion_confint(conversions['treatment'], visitors['treatment'], alpha=0.05, method='normal')[0]
]
ci_upper_values = [
    proportion_confint(conversions['control'], visitors['control'], alpha=0.05, method='normal')[1],
    proportion_confint(conversions['treatment'], visitors['treatment'], alpha=0.05, method='normal')[1]
]

ax2.errorbar(groups, ctr_values, 
             yerr=[[ctr_values[i] - ci_lower_values[i] for i in range(2)], 
                   [ci_upper_values[i] - ctr_values[i] for i in range(2)]],
             fmt='o', capsize=10, color='darkblue', markersize=12)
ax2.set_title('CTR with 95% Confidence Intervals', fontsize=14)
ax2.set_ylabel('Click-Through Rate')
ax2.set_ylim(0, 0.6)
for i, val in enumerate(ctr_values):
    ax2.text(i, val + 0.02, f"{val:.4f}", ha='center')

# Plot 3: Lift
ax3 = axes[1, 0]
lift_data = pd.DataFrame({'Metric': ['Lift'], 'Value': [lift]})
sns.barplot(x='Metric', y='Value', data=lift_data, ax=ax3, palette=['green'])
ax3.set_title(f'Lift (Improvement): {lift:.2f}%', fontsize=14)
ax3.set_ylabel('Percentage Improvement')
ax3.text(0, lift/2, f"{lift:.2f}%", ha='center', fontsize=20)

# Plot 4: Device segmentation
ax4 = axes[1, 1]
device_ctr = df.groupby(['group', 'device_type'])['click'].mean().unstack()
device_ctr.plot(kind='bar', ax=ax4, color=['#FF6B6B', '#4ECDC4'])
ax4.set_title('CTR by Device Type and Group', fontsize=14)
ax4.set_ylabel('CTR')
ax4.set_xlabel('Group')
ax4.legend(title='Device Type')
ax4.set_ylim(0, 0.6)
for i, row in enumerate(device_ctr.iterrows()):
    for j, val in enumerate(row[1]):
        ax4.text(i + j*0.2 - 0.1, val + 0.02, f"{val:.3f}", ha='center')

plt.tight_layout()
plt.savefig('../reports/images/combined_ab_test_results.png', dpi=300, bbox_inches='tight')
plt.close()
print(" Combined figure saved to: ../reports/images/final/combined_ab_test_results.png")

# PART 2: Individual Figures (each plot separately)

# Individual Plot 1: CTR Comparison
fig1, ax1 = plt.subplots(figsize=(8, 6))
sns.barplot(x='group', y='click', data=df, ci=95, palette='Set2')
ax1.set_title('Click-Through Rate by Group with 95% CI', fontsize=14)
ax1.set_ylabel('CTR')
ax1.set_ylim(0, 0.6)
plt.savefig('../reports/images/final/ctr_comparison.png', dpi=300, bbox_inches='tight')
plt.close()
print(" Saved: ctr_comparison.png")

# Individual Plot 2: Confidence Intervals
fig2, ax2 = plt.subplots(figsize=(8, 6))
ax2.errorbar(groups, ctr_values, 
             yerr=[[ctr_values[i] - ci_lower_values[i] for i in range(2)], 
                   [ci_upper_values[i] - ctr_values[i] for i in range(2)]],
             fmt='o', capsize=10, color='darkblue', markersize=12)
ax2.set_title('CTR with 95% Confidence Intervals', fontsize=14)
ax2.set_ylabel('Click-Through Rate')
ax2.set_ylim(0, 0.6)
for i, val in enumerate(ctr_values):
    ax2.text(i, val + 0.02, f"{val:.4f}", ha='center')
plt.savefig('../reports/images/final/confidence_intervals.png', dpi=300, bbox_inches='tight')
plt.close()
print(" Saved: confidence_intervals.png")

# Individual Plot 3: Lift
fig3, ax3 = plt.subplots(figsize=(8, 6))
sns.barplot(x=['Lift'], y=[lift], palette=['green'])
ax3.set_title(f'Lift (Improvement): {lift:.2f}%', fontsize=14)
ax3.set_ylabel('Percentage Improvement')
ax3.text(0, lift/2, f"{lift:.2f}%", ha='center', fontsize=20)
plt.savefig('../reports/images/final/lift_chart.png', dpi=300, bbox_inches='tight')
plt.close()
print(" Saved: lift_chart.png")

# Individual Plot 4: Device Segmentation
fig4, ax4 = plt.subplots(figsize=(8, 6))
device_ctr = df.groupby(['group', 'device_type'])['click'].mean().unstack()
device_ctr.plot(kind='bar', ax=ax4, color=['#FF6B6B', '#4ECDC4'])
ax4.set_title('CTR by Device Type and Group', fontsize=14)
ax4.set_ylabel('CTR')
ax4.set_xlabel('Group')
ax4.legend(title='Device Type')
ax4.set_ylim(0, 0.6)
for i, row in enumerate(device_ctr.iterrows()):
    for j, val in enumerate(row[1]):
        ax4.text(i + j*0.2 - 0.1, val + 0.02, f"{val:.3f}", ha='center')
plt.savefig('../reports/images/final/device_segmentation.png', dpi=300, bbox_inches='tight')
plt.close()
print(" Saved: device_segmentation.png")

print("\n All visualizations saved successfully in '../reports/images/final/'")

## 7. Conclusion and Recommendations
- Business decision
- Next steps

In [ ]:
# Save cleaned dataset for Power BI
cleaned_path = '../data/cleaned_ab_test_data.csv'
df.to_csv(cleaned_path, index=False)
print(f"Cleaned data saved to: {cleaned_path}")

# Export summary results
summary_results = {
    'Metric': ['Control CTR', 'Treatment CTR', 'Lift', 'Z-statistic', 'P-value'],
    'Value': [
        control_ctr,
        treatment_ctr,
        lift,
        z_stat,
        p_value
    ]
}
summary_df = pd.DataFrame(summary_results)
summary_path = '../reports/summary_results.csv'
summary_df.to_csv(summary_path, index=False)
print(f"Summary results saved to: {summary_path}")

# Export device segmentation results
device_results = {
    'Device_Type': ['Mobile', 'Mobile', 'Desktop', 'Desktop'],
    'Group': ['Control', 'Treatment', 'Control', 'Treatment'],
    'CTR': [
        df_mobile[df_mobile['group'] == 'control']['click'].mean(),
        df_mobile[df_mobile['group'] == 'treatment']['click'].mean(),
        df_desktop[df_desktop['group'] == 'control']['click'].mean(),
        df_desktop[df_desktop['group'] == 'treatment']['click'].mean()
    ]
}
device_df = pd.DataFrame(device_results)
device_path = '../reports/device_segmentation_results.csv'
device_df.to_csv(device_path, index=False)
print(f"Device segmentation results saved to: {device_path}")